"# Founder-Departure GitHub Corpus → Repo-Level Survival Dataset\n\nThis demo reproduces `data.py` from the artifact **\"Founder-Departure GitHub Corpus Without Liveness Bias\"**.\n\nThe underlying corpus (`mini_demo_data.json` here, a 14-repo curated subset of the full 67-repo corpus) was mined from the GitHub REST API using historical `created:`/`pushed:` date windows (2011–2015) with **no filter on present-day archived/starred/maintained status** — this avoids the survivorship bias of sampling from \"currently famous\" repo lists.\n\nThis notebook takes that raw per-commit corpus and standardizes it into the `repo_level_founder_departure_survival` dataset: **one example per repo**, with:\n- **input**: JSON-encoded repo/founder features computed strictly *before* the founder's own last commit (no post-departure leakage)\n- **output**: a 3-way survival label — `survived` / `non_surviving` / `unknown_insufficient_post_departure_window`\n\nThe code below is copied nearly verbatim from `data.py`, split into cells with explanatory notes."

In [ ]:
import subprocess, sys\ndef _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])\n\n# data.py itself has zero third-party dependencies (stdlib only: json, logging, collections, datetime, pathlib).\n# The only extra package needed here is for the visualization cell at the end.\nif 'google.colab' not in sys.modules:\n    _pip('matplotlib==3.10.0')

In [ ]:
from __future__ import annotations\n\nimport json\nimport logging\nimport sys\nfrom collections import Counter\nfrom datetime import datetime, timezone\nfrom pathlib import Path\n\nimport matplotlib.pyplot as plt  # for the results visualization cell\n\nlogging.basicConfig(level=logging.INFO, format=\"%(asctime)s %(levelname)s %(message)s\")\nlog = logging.getLogger(\"data\")

## Load the demo data\n\n`mini_demo_data.json` is a 14-repo curated subset of the full raw corpus (`temp/datasets/full_founder_departure_corpus.json` in the original pipeline) — the same schema (`repo_metadata`, `founder_signal`, `commits[]`), just fewer repos and truncated commit lists to keep the file small. We try the GitHub-hosted copy first (for Colab), falling back to the local file."

In [ ]:
GITHUB_DATA_URL = \"https://raw.githubusercontent.com/ai-inventor-papers/ai-invention-24ffbe-pre-departure-bus-factor-diffusion/main/round-2/dataset-1/demo/mini_demo_data.json\"\nimport json, os\n\ndef load_data():\n    try:\n        import urllib.request\n        with urllib.request.urlopen(GITHUB_DATA_URL) as response:\n            return json.loads(response.read().decode())\n    except Exception: pass\n    if os.path.exists(\"mini_demo_data.json\"):\n        with open(\"mini_demo_data.json\") as f: return json.load(f)\n    raise FileNotFoundError(\"Could not load mini_demo_data.json\")

In [ ]:
corpus = load_data()\nrepos = corpus[\"repos\"]\nlog.info(f\"loaded corpus: {len(repos)} repos\")

## Configuration\n\nThe two label thresholds from `data.py` — kept at their **original full-scale values**. They are calendar-day thresholds (not compute-scale knobs like epochs/batch-size), so there is nothing to shrink for a fast demo run: the whole pipeline is a single pass over the (already small) repo list and finishes in well under a second regardless of these values."

In [ ]:
NON_SURVIVAL_STALE_DAYS = 730  # no commit in >=2yr as of build time -> \"non_surviving\" proxy label\nPOST_DEPARTURE_MIN_DAYS_FOR_LABEL = 30  # need at least some post-departure window to call a label at all

## Helper functions\n\n`parse_dt` parses GitHub's ISO-8601 commit timestamps, and `commit_identity` resolves a commit to an author identity (falling back from login → email → name), matching how `founder_signal.dominant_early_author` was computed upstream."

In [ ]:
def parse_dt(s: str | None) -> datetime | None:\n    if not s:\n        return None\n    return datetime.fromisoformat(s.replace(\"Z\", \"+00:00\"))\n\n\ndef commit_identity(c: dict) -> str:\n    return c.get(\"author_login\") or c.get(\"author_email\") or c.get(\"author_name\") or \"unknown\"

## Build the repo-level examples\n\nFor each repo: find the founder's own last commit, restrict the input features to commits at or before that date (the leakage-safe \"pre-departure\" window), then compare the repo's overall last-commit date against `NON_SURVIVAL_STALE_DAYS`/`POST_DEPARTURE_MIN_DAYS_FOR_LABEL` to assign the 3-way label."

In [ ]:
def build_repo_level_examples(repos: list[dict]) -> list[dict]:\n    examples = []\n    label_counts = Counter()\n    for r in repos:\n        meta = r[\"repo_metadata\"]\n        fs = r[\"founder_signal\"]\n        commits = sorted(r[\"commits\"], key=lambda c: c.get(\"date\") or \"\")\n        founder = fs[\"dominant_early_author\"]\n\n        founder_dates = [c[\"date\"] for c in commits if commit_identity(c) == founder and c.get(\"date\")]\n        if not founder_dates:\n            continue\n        founder_last_dt = parse_dt(founder_dates[-1])\n        repo_last_dt = parse_dt(fs[\"last_commit_date\"])\n        if founder_last_dt is None or repo_last_dt is None:\n            continue\n\n        # pre-departure feature window only: commits up to and including the founder's own last commit.\n        # This avoids leaking the post-departure outcome into the input, which would make the label trivial.\n        pre_departure_commits = [c for c in commits if (parse_dt(c.get(\"date\")) or founder_last_dt) <= founder_last_dt]\n        n_contributors_pre = len({commit_identity(c) for c in pre_departure_commits})\n\n        post_departure_days = (repo_last_dt - founder_last_dt).days\n        if post_departure_days < POST_DEPARTURE_MIN_DAYS_FOR_LABEL:\n            label = \"unknown_insufficient_post_departure_window\"\n        else:\n            now = datetime.now(timezone.utc)\n            is_stale = (now - repo_last_dt).days > NON_SURVIVAL_STALE_DAYS\n            label = \"non_surviving\" if is_stale else \"survived\"\n        label_counts[label] += 1\n\n        input_obj = {\n            \"repo_full_name\": meta[\"full_name\"],\n            \"language\": meta[\"language\"],\n            \"repo_created_at\": meta[\"created_at\"],\n            \"founder_last_commit_date\": fs[\"dominant_early_author\"] and founder_dates[-1],\n            \"n_commits_pre_departure\": len(pre_departure_commits),\n            \"n_contributors_pre_departure\": n_contributors_pre,\n            \"dominant_early_author_fraction\": fs[\"dominant_early_author_fraction\"],\n            \"early_window_commit_count\": fs[\"early_window_commit_count\"],\n            \"stargazers_count_at_scrape_time\": meta[\"stargazers_count\"],\n            \"sampling_frame\": meta[\"sampling_frame\"],\n        }\n        examples.append(\n            {\n                \"input\": json.dumps(input_obj, sort_keys=True),\n                \"output\": label,\n                \"metadata_task_type\": \"classification\",\n                \"metadata_n_classes\": 3,\n                \"metadata_repo_full_name\": meta[\"full_name\"],\n                \"metadata_sampling_frame\": meta[\"sampling_frame\"],\n                \"metadata_frame_construction_method\": meta[\"frame_construction_method\"],\n                \"metadata_post_departure_days\": post_departure_days,\n                \"metadata_history_span_years\": meta[\"history_span_years\"],\n                \"metadata_archived\": meta[\"archived\"],\n            }\n        )\n    log.info(f\"repo_level: {len(examples)} examples, label distribution: {dict(label_counts)}\")\n    return examples

## Run the standardization and assemble the final output object\n\nSame structure as the full pipeline's `exp_sel_data_out.json`: a `metadata` block plus a `datasets` list containing the single chosen dataset, `repo_level_founder_departure_survival`."

In [ ]:
repo_examples = build_repo_level_examples(repos)\nassert repo_examples, \"repo_level produced zero examples\"\n\nout = {\n    \"metadata\": {\n        \"source\": \"GitHub REST API, authenticated (GH_TOKEN), liveness-non-conditioned historical search\",\n        \"description\": (\n            \"Repo-level founder-departure survival-prediction view of the liveness_non_conditioned \"\n            \"GitHub corpus built for this artifact: one example per repo, leakage-safe pre-departure \"\n            \"features only, label = survived / non_surviving / unknown_insufficient_post_departure_window.\"\n        ),\n        \"n_source_repos\": len(repos),\n    },\n    \"datasets\": [\n        {\"dataset\": \"repo_level_founder_departure_survival\", \"examples\": repo_examples},\n    ],\n}\n\nprint(f\"Built {len(repo_examples)} repo-level examples from {len(repos)} source repos.\")

## Results\n\nA readable summary table of each repo-level example, plus a bar chart of the label distribution — the question this dataset exists to let downstream analysis answer: *what fraction of founder-departed projects survive?*"

In [ ]:
print(f\"{'repo':40s} {'label':45s} {'post_dep_days':>14s} {'span_yrs':>9s}\")\nprint(\"-\" * 112)\nfor ex in repo_examples:\n    print(\n        f\"{ex['metadata_repo_full_name']:40s} {ex['output']:45s} \"\n        f\"{ex['metadata_post_departure_days']:14d} {ex['metadata_history_span_years']:9.2f}\"\n    )\n\nlabel_counts = Counter(ex[\"output\"] for ex in repo_examples)\nlabels = list(label_counts.keys())\ncounts = [label_counts[l] for l in labels]\n\nfig, ax = plt.subplots(figsize=(7, 4))\nax.bar(labels, counts, color=[\"#4C72B0\", \"#DD8452\", \"#999999\"][: len(labels)])\nax.set_ylabel(\"# repos\")\nax.set_title(f\"Founder-departure survival label distribution (n={len(repo_examples)} repos, demo subset)\")\nplt.xticks(rotation=20, ha=\"right\")\nplt.tight_layout()\nplt.show()\n\nprint(\"\\nLabel counts:\", dict(label_counts))